# 10c -- MTMC Stages 4-5: Association + Evaluation

**Prerequisite**: attach **10b's output** as a data source:
`Add Data -> Kernel Output -> search "mtmc-10b-stage-3-faiss-indexing" -> add`

**This is the iteration loop** -- edit the tuning params cell and re-run in ~6 min.
No GPU needed, no data download, no model inference.

| Stage | What | Time |
|---|---|---|
| 4 | Cross-camera association (AQE + Louvain graph clustering) | ~5 min |
| 5 | Evaluation: IDF1, MOTA, HOTA | ~1 min |

In [ ]:
import os, sys, subprocess, shutil, json, time, tarfile
from pathlib import Path
import torch

print(f"Python : {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA   : {torch.cuda.is_available()}")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}  ({p.total_memory/1024**3:.1f} GB)")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nUsing device: {DEVICE}")

## 1. Clone Repo & Install Dependencies

In [ ]:
REPO_URL = "https://github.com/MRKDaGods/gp.git"
WORK_DIR = Path("/kaggle/working")
PROJECT  = WORK_DIR / "gp"

if not PROJECT.exists():
    print(f"Cloning {REPO_URL} ...")
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT)])
else:
    print("Repo already present, pulling latest ...")
    subprocess.check_call(["git", "-C", str(PROJECT), "pull", "--ff-only"])

os.chdir(str(PROJECT))
sys.path.insert(0, str(PROJECT))
print(f"\n\u2713 Repo ready at {PROJECT}")

In [ ]:
def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

try:
    import faiss; print(f"faiss ok ({faiss.__version__})")
except ImportError:
    try: pip("faiss-gpu")
    except Exception: pip("faiss-cpu")

try:
    import trackeval; print("trackeval ok")
except ImportError:
    pip("git+https://github.com/JonathonLuiten/TrackEval.git")

pip("motmetrics", "loguru", "omegaconf", "rich", "networkx>=3.1", "click",
    "numpy", "scipy", "pandas", "scikit-learn")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps", "-q"],
                      cwd=str(PROJECT))
print("\n\u2713 Dependencies installed")

In [ ]:
FAILED = []
_checks = [
    ("faiss", "faiss"),
    ("motmetrics", "motmetrics"),
    ("loguru", "loguru"),
    ("omegaconf", "omegaconf"),
    ("networkx", "networkx"),
    ("sklearn", "sklearn"),
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("trackeval", "trackeval"),
]
for label, mod in _checks:
    try:
        __import__(mod)
        print(f"  \u2713 {label}")
    except ImportError as e:
        print(f"  \u2717 {label}: {e}")
        FAILED.append(label)
if FAILED:
    raise RuntimeError(f"Missing modules: {FAILED} -- fix pip installs above")
print("\n\u2713 All required modules importable")

## 2. Load Checkpoint from 10b

Finds `checkpoint.tar.gz` in `/kaggle/input/mtmc-10b-stage-3-faiss-indexing/` and extracts it.

In [ ]:
PREV_SLUG = "mtmc-10b-stage-3-faiss-indexing"
PREV_INPUT = Path("/kaggle/input") / PREV_SLUG

if not PREV_INPUT.exists():
    for p in Path("/kaggle/input").iterdir():
        if PREV_SLUG in p.name or "stage3" in p.name or "10b" in p.name:
            PREV_INPUT = p; break

cp = PREV_INPUT / "checkpoint.tar.gz"

# Fallback: download via Kaggle API if not found in mounted input
if not cp.exists():
    print(f"checkpoint.tar.gz not found at {cp} -- attempting API download")
    import subprocess as _sp
    _dl_dir = Path("/tmp/kaggle_dl")
    _dl_dir.mkdir(parents=True, exist_ok=True)
    _r = _sp.run(
        ["kaggle", "kernels", "output",
         "mrkdagods/mtmc-10b-stage-3-faiss-indexing",
         "--file", "checkpoint.tar.gz",
         "-p", str(_dl_dir)],
        capture_output=True, text=True
    )
    print(_r.stdout); print(_r.stderr)
    cp = _dl_dir / "checkpoint.tar.gz"

assert cp.exists(), f"checkpoint.tar.gz not found at {cp}"

EXTRACT_DIR = Path("/tmp/pipeline_run")
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Extracting {cp.stat().st_size/1024**2:.1f} MB ...")
with tarfile.open(str(cp), "r:gz") as tar:
    tar.extractall(str(EXTRACT_DIR))

with open(EXTRACT_DIR / "run_metadata.json") as f:
    meta = json.load(f)
RUN_NAME = meta["run_name"]
DATA_OUT = EXTRACT_DIR
RUN_DIR  = EXTRACT_DIR / RUN_NAME
# GT annotations are in the repo under data/raw/cityflowv2/*/gt/gt.txt
# They were force-committed despite data/ being gitignored.
_repo_gt = PROJECT / "data" / "raw" / "cityflowv2"
if any((_repo_gt / cam / "gt" / "gt.txt").exists()
       for cam in ["S01_c001", "S01_c002", "S01_c003",
                   "S02_c006", "S02_c007", "S02_c008"]):
    GT_DIR = str(_repo_gt)
    print(f"\u2713 GT annotations at {GT_DIR}")
else:
    GT_DIR = ""
    print("WARNING: GT files not found in repo — eval will skip metrics")
print(f"\u2713 Checkpoint extracted -- run: {RUN_NAME}")
for s in ["stage1", "stage2", "stage3"]:
    d = RUN_DIR / s
    if d.exists(): print(f"  {s}: {len(list(d.rglob('*')))} files")
print(f"  GT dir: {GT_DIR}")

## 3. Tuning Parameters

**Edit these values** then re-run the cells below (~6 min). No need to re-run 10a or 10b.

In [ ]:
# ---- Stage 4: Cross-camera association (v43 optimized) ---------------------
# v43: 6 rounds of sweeps found optimal config:
#   threshold=0.53, FAC enabled (+0.7pp),
#   AQE k=3 (fewer conflated IDs), intra-camera merge (+1.0pp),
#   sec_weight=0.25 (+0.1pp)
#   Total improvement: +2.5pp IDF1 (0.789 -> 0.814 local)
# v46: FIC covariance bug fix -- reg shifted from 0.15 -> 3.0 (+0.25pp)
SIM_THRESH = 0.53
FIC_REG    = 3.0

# OSNet ensemble: point to secondary embeddings from 10a
SECONDARY_EMB     = str(RUN_DIR / "stage2" / "embeddings_secondary.npy")
SECONDARY_WEIGHT  = 0.25   # v43: 0.25 optimal (0.30->0.25 = +0.1pp)

# ---- Stage 5: Evaluation ----------------------------------------------------
# v42: MTMC_ONLY = False.  Single-cam trajectories are NOT guaranteed FP
# in the MTMC eval -- they contribute IDTP for GT vehicles correctly
# identified in one camera.  Dropping them costs ~5pp IDF1.
MTMC_ONLY = False

# v45: Stationary vehicle filter (+1.4pp IDF1 local: 0.814 -> 0.828)
# Removes trajectories where ALL tracklets have < min_displacement_px bbox displacement.
# d150 is optimal: removes ~22% of pred rows without losing true positives.
STATIONARY_FILTER_ENABLED = True
STATIONARY_FILTER_MIN_DISP = 150

import os as _os
has_secondary = _os.path.exists(SECONDARY_EMB)
print(f"Stage 4: sim_thresh={SIM_THRESH}, fic_reg={FIC_REG}")
print(f"  secondary: {'FOUND' if has_secondary else 'NOT FOUND'}")
print(f"  secondary_weight: {SECONDARY_WEIGHT}")
print(f"Stage 5: mtmc_only={MTMC_ONLY}")
print(f"  stationary_filter: enabled={STATIONARY_FILTER_ENABLED}, min_disp={STATIONARY_FILTER_MIN_DISP}px")

## 4. Run Stages 4-5

In [ ]:
os.chdir(str(PROJECT))
cmd = [
    sys.executable, "scripts/run_pipeline.py",
    "--config", "configs/default.yaml",
    "--dataset-config", "configs/datasets/cityflowv2.yaml",
    "--stages", "4,5",
    "--override", f"project.run_name={RUN_NAME}",
    "--override", f"project.output_dir={DATA_OUT}",
    "--override", "stage0.cameras=[S01_c001,S01_c002,S01_c003,S02_c006,S02_c007,S02_c008]",
    # Stage 4 optimized (v43 sweep-tuned)
    "--override", f"stage4.association.graph.similarity_threshold={SIM_THRESH}",
    "--override", f"stage4.association.fic.regularisation={FIC_REG}",
    # v43: FAC cross-camera feature augmentation (+0.7pp)
    "--override", "stage4.association.fac.enabled=true",
    "--override", "stage4.association.fac.knn=20",
    "--override", "stage4.association.fac.learning_rate=0.5",
    "--override", "stage4.association.fac.beta=0.08",
    # v46: AQE k=2 optimal after QE self-exclusion fix
    "--override", "stage4.association.query_expansion.k=2",
    # v43: intra-camera merge (+1.0pp)
    "--override", "stage4.association.intra_camera_merge.enabled=true",
    "--override", "stage4.association.intra_camera_merge.threshold=0.75",
    "--override", "stage4.association.intra_camera_merge.max_time_gap=60",
    "--override", "stage4.association.fic.enabled=true",
    # Stage 5 settings
    "--override", f"stage5.mtmc_only_submission={str(MTMC_ONLY).lower()}",
    "--override", "stage5.gt_frame_clip=true",
    "--override", "stage5.gt_zone_filter=true",
    "--override", "stage5.gt_zone_margin_frac=0.2",
    "--override", "stage5.gt_frame_clip_min_iou=0.5",
    # v45: Stationary vehicle filter (+1.4pp IDF1)
    "--override", f"stage5.stationary_filter.enabled={str(STATIONARY_FILTER_ENABLED).lower()}",
    "--override", f"stage5.stationary_filter.min_displacement_px={STATIONARY_FILTER_MIN_DISP}",
]
# Add secondary embeddings if available
if has_secondary:
    cmd += [
        "--override", f"stage4.association.secondary_embeddings.path={SECONDARY_EMB}",
        "--override", f"stage4.association.secondary_embeddings.weight={SECONDARY_WEIGHT}",
    ]
if GT_DIR:
    cmd += ["--override", f"stage5.ground_truth_dir={GT_DIR}"]
else:
    print("WARNING: GT_DIR is empty -- eval will skip metric computation")
print("CMD:", " ".join(str(c) for c in cmd))
print("=" * 70)
t0 = time.time()
r = subprocess.run(cmd, cwd=str(PROJECT))
print("=" * 70)
elapsed = time.time() - t0
if r.returncode != 0:
    print(f"FAILED after {elapsed/60:.1f} min"); sys.exit(r.returncode)
print(f"Stages 4-5 done in {elapsed/60:.1f} min")

## 5. Results

In [ ]:
stage5_dir = RUN_DIR / "stage5"

def _pct(v):
    return f"{v:.1%}" if isinstance(v, float) else str(v)

metrics_files = sorted(stage5_dir.glob("metrics_*.json")) if stage5_dir.exists() else []
if metrics_files:
    print("=" * 65)
    print("EVALUATION RESULTS")
    print("=" * 65)
    for mf in metrics_files:
        m = json.loads(mf.read_text())
        m = m.get("metrics", m)
        cam = mf.stem.replace("metrics_", "")
        idf1 = _pct(m.get("IDF1", m.get("idf1", "?")))
        mota = _pct(m.get("MOTA", m.get("mota", "?")))
        hota = _pct(m.get("HOTA", m.get("hota", "?")))
        idsw = m.get("ID_Sw", m.get("id_switches", "?"))
        print(f"  {cam:12s}  IDF1={idf1}  MOTA={mota}  HOTA={hota}  IDsw={idsw}")

for fname in ["summary.json", "evaluation_report.json"]:
    sf = stage5_dir / fname
    if sf.exists():
        s = json.loads(sf.read_text())
        print("-" * 65 + "\n  GLOBAL:")
        for k in ["IDF1","MOTA","HOTA","ID_Sw","idf1","mota","hota","id_switches","mtmc_idf1"]:
            v = s.get(k)
            if v is not None: print(f"    {k}: {_pct(v)}")
        break

if not metrics_files:
    print("No metrics files found -- check stage5 output dir:", stage5_dir)

# Copy evaluation report to /kaggle/working/ for easy download
import shutil as _shutil
for _fname in ["evaluation_report.json", "summary.json"]:
    _src = stage5_dir / _fname
    if _src.exists():
        _shutil.copy2(str(_src), str(Path("/kaggle/working") / _fname))
        print(f"Copied {_fname} to /kaggle/working/")


## 6. Automated Parameter Scan (optional)

Runs stages 4-5 across a grid of parameter values and reports the best combination.
Each combination takes ~2 min → a 12-point scan takes ~24 min total.

**Comment out this cell if you just want the single run above.**

In [ ]:
# ============================================================
# Phase 3 Scan: Test temporal_split + CamTTA with 384px features
# SET SCAN_ENABLED = True to run the grid search.
# ============================================================
SCAN_ENABLED = False

if SCAN_ENABLED:
    import itertools

    # Phase 3 fixed optimal baseline config (v48 best result)
    P3_SIM_THRESH  = 0.53   # optimal from v46 sweep
    P3_FIC_REG     = 3.0    # optimal FIC regularisation
    P3_AQE_K       = 2      # optimal after QE self-exclusion fix

    # Phase 3 scan grid: temporal_split + CamTTA combos
    # ~24 combinations, ~35-45 min on Kaggle T4
    scan_grid = {
        "temporal_split":          [False, True],
        "split_threshold":         [0.40, 0.45, 0.50],
        "camera_tta":              [False, True],
        "sim_thresh":              [0.51, 0.53, 0.55],
        "use_secondary":           [True],  # keep OSNet ensemble
    }

    keys   = list(scan_grid.keys())
    combos = list(itertools.product(*[scan_grid[k] for k in keys]))
    # Filter: split_threshold only matters when temporal_split=True
    combos = [c for c in combos if dict(zip(keys, c))["temporal_split"] or dict(zip(keys, c))["split_threshold"] == 0.45]
    print(f"Phase 3 scan: {len(combos)} combinations ...")

    results = []
    for combo in combos:
        params = dict(zip(keys, combo))
        ts     = params["temporal_split"]
        st     = params["split_threshold"]
        ctta   = params["camera_tta"]
        st_w   = P3_SIM_THRESH if params["sim_thresh"] is None else params["sim_thresh"]
        sec    = params["use_secondary"]

        scan_run = (f"scan_ts{int(ts)}_st{int(st*100)}"
                    f"_ctta{int(ctta)}_sim{int(st_w*100)}")
        (DATA_OUT / scan_run).mkdir(parents=True, exist_ok=True)

        cmd_scan = [
            sys.executable, "scripts/run_pipeline.py",
            "--config", "configs/default.yaml",
            "--dataset-config", "configs/datasets/cityflowv2.yaml",
            "--stages", "4,5",
            "--override", f"project.run_name={scan_run}",
            "--override", f"project.output_dir={DATA_OUT}",
            "--override", "stage0.cameras=[S01_c001,S01_c002,S01_c003,S02_c006,S02_c007,S02_c008]",
            "--override", f"stage4.association.graph.similarity_threshold={st_w}",
            "--override", f"stage4.association.fic.regularisation={P3_FIC_REG}",
            "--override", "stage4.association.fac.enabled=true",
            "--override", "stage4.association.fac.knn=20",
            "--override", "stage4.association.fac.learning_rate=0.5",
            "--override", "stage4.association.fac.beta=0.08",
            "--override", f"stage4.association.query_expansion.k={P3_AQE_K}",
            "--override", "stage4.association.intra_camera_merge.enabled=true",
            "--override", "stage4.association.intra_camera_merge.threshold=0.75",
            "--override", "stage4.association.intra_camera_merge.max_time_gap=60",
            "--override", "stage4.association.fic.enabled=true",
            "--override", "stage4.association.reranking.enabled=false",
            "--override", f"stage4.association.temporal_split.enabled={str(ts).lower()}",
            "--override", f"stage4.association.temporal_split.split_threshold={st}",
            "--override", f"stage2.camera_tta.enabled={str(ctta).lower()}",
            "--override", f"stage5.mtmc_only_submission={str(MTMC_ONLY).lower()}",
            "--override", "stage5.gt_frame_clip=true",
            "--override", "stage5.gt_frame_clip_min_iou=0.5",
            "--override", "stage5.gt_zone_filter=true",
            "--override", "stage5.gt_zone_margin_frac=0.2",
        ]
        if sec and has_secondary:
            sec_emb = str(RUN_DIR / "stage2" / "embeddings_secondary.npy")
            cmd_scan += [
                "--override", f"stage4.association.secondary_embeddings.path={sec_emb}",
                "--override", f"stage4.association.secondary_embeddings.weight=0.25",
            ]
        if GT_DIR:
            cmd_scan += ["--override", f"stage5.ground_truth_dir={GT_DIR}"]

        t0 = time.time()
        r  = subprocess.run(cmd_scan, cwd=str(PROJECT), capture_output=True)
        elapsed = time.time() - t0

        report = DATA_OUT / scan_run / "stage5" / "evaluation_report.json"
        idf1 = mota = hota = 0.0
        if report.exists():
            rp = json.loads(report.read_text())
            m  = rp.get("metrics", rp)
            idf1 = m.get("mtmc_idf1", m.get("IDF1", m.get("idf1", 0.0)))
            mota = m.get("MOTA", m.get("mota", 0.0))
            hota = m.get("HOTA", m.get("hota", 0.0))

        results.append({**params, "IDF1": idf1, "MOTA": mota, "HOTA": hota, "time": elapsed})
        status = "OK" if r.returncode == 0 else "FAIL"
        print(f"  [{status}] ts={int(ts)} st={st} ctta={int(ctta)} sim={st_w} -> IDF1={idf1:.4f} ({elapsed:.0f}s)")

    has_hota = any(r2["HOTA"] > 0 for r2 in results)
    sort_key = "HOTA" if has_hota else "IDF1"
    results.sort(key=lambda x: x[sort_key], reverse=True)
    print()
    print("=" * 80)
    print(f"PHASE 3 SCAN RESULTS (sorted by {sort_key})")
    print("=" * 80)
    for idx, r2 in enumerate(results[:20]):
        print(f"  #{idx+1}: ts={int(r2['temporal_split'])} st={r2['split_threshold']} ctta={int(r2['camera_tta'])} sim={r2['sim_thresh']} -> IDF1={r2['IDF1']:.4f} MOTA={r2['MOTA']:.4f} HOTA={r2['HOTA']:.4f}")
    best = results[0]
    print(f"\nBEST: ts={int(best['temporal_split'])} st={best['split_threshold']} ctta={int(best['camera_tta'])} sim={best['sim_thresh']} -> IDF1={best['IDF1']:.4f}")

    import json as _json
    results_path = DATA_OUT / "scan_phase3_results.json"
    with open(results_path, "w") as _f:
        _json.dump({"sort_key": sort_key, "results": results}, _f, indent=2)
    import shutil as _shutil
    _shutil.copy2(str(results_path), "/kaggle/working/scan_phase3_results.json")
    print("Scan results saved to /kaggle/working/scan_phase3_results.json")
else:
    print("Phase 3 scan disabled. Set SCAN_ENABLED = True to run.")
